In [1]:
!nvidia-smi
!pip install ultralytics --quiet
!pip install opencv-python-headless --quiet

import ultralytics
ultralytics.checks()
from google.colab import drive
drive.mount('/content/drive')

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 42.9/235.7 GB disk)
Mounted at /content/drive


In [2]:
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/Training_data_10p.zip'
EXTRACT_DIR = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Done.')

# Verify structure
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

Extracting...
Done.
dataset/
  Training_data_10p/
    data_10_full.yaml
    dataset/
      labels/
        val/
        train/
      images/
        val/
        train/


In [3]:
import yaml

dataset_config = {
    'path': '/content/dataset/Training_data_10p/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc': 1,
    'names': ['car']
}

yaml_path = '/content/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(open(yaml_path).read())

names:
- car
nc: 1
path: /content/dataset/Training_data_10p/dataset
train: images/train
val: images/val



In [4]:
import cv2
import matplotlib.pyplot as plt
import random
import glob

def draw_yolo_boxes(img_path, label_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
    return img

image_files = glob.glob('/content/dataset/Training_data_10p/dataset/images/train/**/*.jpg', recursive=True) + \
              glob.glob('/content/dataset/Training_data_10p/dataset/images/train/**/*.png', recursive=True)

samples = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flatten(), samples):
    label_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'
    img = draw_yolo_boxes(img_path, label_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Total training images: {len(image_files)}')

Output hidden; open in https://colab.research.google.com to view.

In [5]:
from ultralytics import YOLO
import pandas as pd
import os

# ── Configuration ─────────────────────────────────────────────────────────────
YAML_PATH = yaml_path
BASE_PROJECT = '/content/runs/ablation'
EPOCHS = 10
IMGSZ = 640
DEVICE = 0
DRIVE_OUTPUT = '/content/drive/MyDrive/D7047E Project Datasets/yolov9c_results/ablation'

# Fixed parameters across all runs
FIXED = dict(
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    batch=32,
    workers=4,
    pretrained=True,
    freeze=10,          # frozen backbone — best from previous experiments
    patience=EPOCHS,   # disable early stopping for fair comparison
    save=True,
    plots=False,
    cache='disk',
    amp=True,
    project=BASE_PROJECT,
    exist_ok=True,
    # Fixed augmentation parameters (thesis values)
    translate=0.1,
    mosaic=1.0,
    lr0=0.01,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
)

# ── OAT Experiment definitions ────────────────────────────────────────────────
# Baseline = thesis values: hsv_s=0.7, degrees=45, scale=0.9
# One parameter varied at a time, others fixed at thesis value

experiments = [
    # Baseline — all thesis values
    dict(name='baseline',      hsv_s=0.7, hsv_h=0.015, hsv_v=0.4, degrees=45.0, scale=0.9),
    # hsv_s sweep (saturation)
    dict(name='hsvs_low',      hsv_s=0.3, hsv_h=0.015, hsv_v=0.4, degrees=45.0, scale=0.9),
    dict(name='hsvs_mid',      hsv_s=0.5, hsv_h=0.015, hsv_v=0.4, degrees=45.0, scale=0.9),
    # degrees sweep (rotation)
    dict(name='degrees_low',   hsv_s=0.7, hsv_h=0.015, hsv_v=0.4, degrees=15.0, scale=0.9),
    dict(name='degrees_mid',   hsv_s=0.7, hsv_h=0.015, hsv_v=0.4, degrees=30.0, scale=0.9),
    # scale sweep (altitude simulation)
    dict(name='scale_low',     hsv_s=0.7, hsv_h=0.015, hsv_v=0.4, degrees=45.0, scale=0.5),
    dict(name='scale_mid',     hsv_s=0.7, hsv_h=0.015, hsv_v=0.4, degrees=45.0, scale=0.7),
]

# ── Run all experiments ───────────────────────────────────────────────────────
results_log = []
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running: {exp['name']} | hsv_s={exp['hsv_s']} degrees={exp['degrees']} scale={exp['scale']}")
    print(f"{'='*60}")

    model = YOLO('yolov9c.pt')

    model.train(
        **FIXED,
        name=exp['name'],
        hsv_h=exp['hsv_h'],
        hsv_s=exp['hsv_s'],
        hsv_v=exp['hsv_v'],
        degrees=exp['degrees'],
        scale=exp['scale'],
    )

    # Extract best epoch metrics
    results_csv = f"{BASE_PROJECT}/{exp['name']}/results.csv"
    if os.path.exists(results_csv):
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
        entry = {
            'experiment':   exp['name'],
            'hsv_s':        exp['hsv_s'],
            'degrees':      exp['degrees'],
            'scale':        exp['scale'],
            'best_epoch':   int(best_row['epoch']),
            'mAP50':        round(best_row['metrics/mAP50(B)'], 4),
            'mAP50-95':     round(best_row['metrics/mAP50-95(B)'], 4),
            'precision':    round(best_row['metrics/precision(B)'], 4),
            'recall':       round(best_row['metrics/recall(B)'], 4),
            'val_box_loss': round(best_row['val/box_loss'], 4),
        }
        results_log.append(entry)

        # Save individual CSV to Drive
        import shutil
        shutil.copy(results_csv, f"{DRIVE_OUTPUT}/{exp['name']}_results.csv")
        print(f"  Best mAP50: {entry['mAP50']} at epoch {entry['best_epoch']}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("ABLATION SUMMARY")
print(f"{'='*60}")
summary_df = pd.DataFrame(results_log)
print(summary_df.to_string(index=False))
summary_df.to_csv(f'{DRIVE_OUTPUT}/ablation_summary.csv', index=False)
print(f"\nSummary saved to {DRIVE_OUTPUT}/ablation_summary.csv")



Running: baseline | hsv_s=0.7 degrees=45.0 scale=0.9
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9c.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=False, opset=None, optimize=

In [6]:
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/D7047E Project Datasets/yolov9c_results/ablation'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Save all individual run results
experiments_names = ['baseline', 'hsvs_low', 'hsvs_mid', 'degrees_low', 'degrees_mid', 'scale_low', 'scale_mid']

for name in experiments_names:
    src_csv = f'/content/runs/ablation/{name}/results.csv'
    src_best = f'/content/runs/ablation/{name}/weights/best.pt'
    if os.path.exists(src_csv):
        shutil.copy(src_csv, f'{DRIVE_OUTPUT}/{name}_results.csv')
    if os.path.exists(src_best):
        shutil.copy(src_best, f'{DRIVE_OUTPUT}/{name}_best.pt')

print('All saved.')

All saved.
